## Cargar Datos

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Cargar dataSet
dfVuelos = pd.read_csv("flights.csv")

In [3]:
# Ensuciar el dataset para incluir nulos en columnas numéricas clave
def ensuciar_df(df, columnas, porcentaje=0.05):
    for col in columnas:
        mask = np.random.random(len(df)) < porcentaje
        df.loc[mask, col] = np.nan

np.random.seed(42)
ensuciar_df(dfVuelos, columnas=["duration", "days_left"], porcentaje=0.05)

## Preparar conjuntos de entrenamiento y pruebas

In [4]:
# Categorizar el campo price_cat para estratificar los sets de training y test

# Distribución de categorías de precio
# Crear categorías de precio (bajo/medio/alto)
dfVuelos['price_cat'] = pd.cut(dfVuelos['price'],
                                bins=[0, 10000, 30000, np.inf],
                                labels=['bajo', 'medio', 'alto'])

dfVuelos

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,price_cat
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1.0,5953,bajo
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1.0,5953,bajo
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1.0,5956,bajo
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1.0,5955,bajo
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1.0,5955,bajo
...,...,...,...,...,...,...,...,...,...,...,...,...
300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,NaN,49.0,69265,alto
300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49.0,77105,alto
300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,NaN,79099,alto
300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49.0,81585,alto


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
# Generando los dataSets estratificados

# División estratificada por precio
strat_train_set, strat_test_set = train_test_split(dfVuelos, 
                                                     test_size=0.2, 
                                                     stratify=dfVuelos["price_cat"], 
                                                     random_state=42)

print('Tamaño del strat_train_set', strat_train_set.shape)
print('Tamaño del strat_test_set', strat_test_set.shape)

Tamaño del strat_train_set (240122, 12)
Tamaño del strat_test_set (60031, 12)


In [7]:
# Prepara el dataSet de training
# Separar los atributos de la variable objetivo

# Eliminar la columna price_cat de los sets de entrenamiento y prueba
# (Esta columna solo se usó para hacer el muestreo estratificado)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("price_cat", axis=1, inplace=True)

## Preparar transformadores y pipeline para preprocesamiento

In [8]:
# Bibliotecas para crear el transformador de clusters
 
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

In [9]:
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
 
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  
 
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [10]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, OrdinalEncoder

# Función para calcular ratios entre dos columnas
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )

# Pipeline logarítmico: para variables numéricas con sesgo positivo
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

# ClusterSimilarity sobre duration y days_left
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
cluster_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    cluster_simil
)

# Pipeline numérico por defecto
default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# Pipeline categórico 1: OneHotEncoder para variables con <= 7 categorías
# airline(6), source_city(6), departure_time(6), stops(3),
# arrival_time(6), destination_city(6), class(2)
cat_attribs_ohe = ["airline", "source_city", "departure_time", "stops",
                   "arrival_time", "destination_city", "class"]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

# Pipeline categórico 2: OrdinalEncoder para 'flight' (1561 categorías únicas)
# OrdinalEncoder mapea cada categoría a un entero sin generar alta dimensionalidad
cat_attribs_ord = ["flight"]

cat_pipeline2 = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
)

# Pipeline completo
preprocessing = ColumnTransformer([
    ("ratio_dur_days", ratio_pipeline(), ["duration", "days_left"]),
    ("log", log_pipeline, ["duration", "days_left"]),
    ("geo_clusters", cluster_pipeline, ["duration", "days_left"]),
    ("cat", cat_pipeline, cat_attribs_ohe),
    ("cat2", cat_pipeline2, cat_attribs_ord),
],
remainder=default_num_pipeline)

In [11]:
X = preprocessing.fit_transform(strat_train_set)

In [12]:
dfFlightsTrainProc = pd.DataFrame(
    X,
    columns=preprocessing.get_feature_names_out(),
    index=strat_train_set.index
)

dfFlightsTrainProc.head(2)

,ratio_dur_days__ratio,log__duration,log__days_left,geo_clusters__Cluster 0 similarity,geo_clusters__Cluster 1 similarity,geo_clusters__Cluster 2 similarity,geo_clusters__Cluster 3 similarity,geo_clusters__Cluster 4 similarity,geo_clusters__Cluster 5 similarity,geo_clusters__Cluster 6 similarity,...,cat__destination_city_Bangalore,cat__destination_city_Chennai,cat__destination_city_Delhi,cat__destination_city_Hyderabad,cat__destination_city_Kolkata,cat__destination_city_Mumbai,cat__class_Business,cat__class_Economy,cat2__flight,remainder__price
17940,-0.118147,1.370661,0.818913,1.955487e-136,3.152485e-142,0.0,1.157528e-13,7.688334e-152,0.0,1.850000e-283,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,710.0,-0.649430
206492,-0.206708,1.261020,1.070754,2.386726e-224,1.336220e-161,0.0,3.623950e-16,7.077032e-109,0.0,0.000000e+00,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1472.0,-0.655291
